# 🚀 CLSS — Train & Test trên Subset 25%

**Fix OOM**: `in_memory: false` → dataset đọc từ disk thay vì load toàn bộ vào RAM.

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists('/kaggle/working/CLSS'):
    !git clone https://github.com/thanhxuan217/CLSS.git
else:
    print('CLSS already cloned')
%cd /kaggle/working/CLSS
!pip install -r requirements.txt --upgrade -q

## Cấu hình & Kiểm tra

In [ ]:
import os

# ============================================================================
# ĐƯỜNG DẪN — CHỈNH SỬA NẾU CẦN
# ============================================================================
VIEW1_PATH = "/kaggle/input/datasets/xunhunh/subsettxt/data/sino_nom_punct_subset"
VIEW2_PATH = "/kaggle/input/datasets/xunhunh/subsettxt/data/sino_nom_punct_doc_subset"
SIKUBERT_PATH = "/kaggle/input/models/xunhunh/sikubertlocal/pytorch/default/1/pretrained/sikubert"
OUT_DIR = "/kaggle/working/output"

# Kiểm tra
for name, path in [("View 1", VIEW1_PATH), ("View 2", VIEW2_PATH), ("SikuBERT", SIKUBERT_PATH)]:
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌'} {name}: {path}")
    if ok and os.path.isdir(path):
        for fn in sorted(os.listdir(path)):
            fp = os.path.join(path, fn)
            if os.path.isfile(fp):
                print(f"       {fn}: {os.path.getsize(fp)/1024:.0f} KB")

## Patch config YAML

Chỉ thay thế transformer model, tag_dictionary, và target_dir.

Đường dẫn data đã đúng sẵn trong config. **Không replace substring `data/` để tránh path bị nhân đôi.**

In [ ]:
import yaml

src = "config/sino_nom_doc_cl_subset.yaml"
dst = "config/sino_nom_doc_cl_kaggle.yaml"

with open(src, "r", encoding="utf-8") as f:
    content = f.read()

# Chỉ thay thế những gì CẦN thay — KHÔNG replace data paths
# (vì config đã có đường dẫn Kaggle đúng rồi)
replacements = {
    "model: SIKU-BERT/sikubert": f"model: {SIKUBERT_PATH}",
    "tag_dictionary: resources/taggers/sino_nom_punct_tags.pkl": f"tag_dictionary: /kaggle/working/CLSS/resources/taggers/sino_nom_punct_tags.pkl",
    "target_dir: resources/taggers/": f"target_dir: {OUT_DIR}/",
}

for old, new in replacements.items():
    if old in content:
        content = content.replace(old, new)
        print(f"  ✅ Replaced: {old[:50]}...")
    else:
        print(f"  ⏭️  Not found (already set?): {old[:50]}...")

with open(dst, "w", encoding="utf-8") as f:
    f.write(content)

# Verify
with open(dst, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

v1 = cfg['punct']['ColumnCorpus-SinoNomPunct']
v2 = cfg['punct']['ColumnCorpus-SinoNomPunctDoc']
print(f"\n📝 Verified config:")
print(f"  View 1 data:   {v1['data_folder']}")
print(f"  View 1 in_mem: {v1.get('in_memory', 'NOT SET ⚠️')}")
print(f"  View 2 data:   {v2['data_folder']}")
print(f"  View 2 in_mem: {v2.get('in_memory', 'NOT SET ⚠️')}")
print(f"  Transformer:   {cfg['embeddings']['TransformerWordEmbeddings-0']['model']}")
print(f"  col_format:    {v1['column_format']} (keys: {[type(k).__name__ for k in v1['column_format']]}")
print(f"  Output:        {cfg['target_dir']}")

# Sanity checks
assert all(isinstance(k, int) for k in v1['column_format']), "column_format keys must be int!"
assert v1.get('in_memory') == False, "in_memory must be false!"
assert '/kaggle/input/' not in str(v1['data_folder']).replace('/kaggle/input/', '', 1), \
    f"Path doubled! Got: {v1['data_folder']}"
print("\n✅ All checks passed!")

## 🚀 TRAINING

In [ ]:
import subprocess, sys

print(f"🚀 Training (10 epochs, subset 25%)")
print(f"   Model:  {SIKUBERT_PATH}")
print(f"   Output: {OUT_DIR}")
print("=" * 60, flush=True)

retcode = subprocess.call([
    sys.executable, 'train.py',
    '--config', 'config/sino_nom_doc_cl_kaggle.yaml',
    '--transformer_model', SIKUBERT_PATH,
    '--out_dir', OUT_DIR
])
print(f'\n{"✅" if retcode == 0 else "❌"} Training finished (code: {retcode})')

## 📊 EVALUATION

In [ ]:
import subprocess, sys

print("📊 Evaluating on test set...")
print("=" * 60, flush=True)

retcode = subprocess.call([
    sys.executable, 'train.py',
    '--config', 'config/sino_nom_doc_cl_kaggle.yaml',
    '--test',
    '--transformer_model', SIKUBERT_PATH,
    '--out_dir', OUT_DIR
])
print(f'\n{"✅" if retcode == 0 else "❌"} Evaluation finished (code: {retcode})')

## Xem kết quả

In [ ]:
import os, glob

print("📁 Output files:")
print("=" * 60)
if os.path.exists(OUT_DIR):
    for root, dirs, files in os.walk(OUT_DIR):
        level = root.replace(OUT_DIR, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        for file in sorted(files):
            fp = os.path.join(root, file)
            print(f'{indent}  {file} ({os.path.getsize(fp)/1024/1024:.1f} MB)')

log_file = os.path.join(OUT_DIR, 'training.log')
if os.path.exists(log_file):
    print("\n📋 Training log (last 50 lines):")
    print("=" * 60)
    with open(log_file, 'r') as f:
        for line in f.readlines()[-50:]:
            print(line.rstrip())